# NB07 — Residual SAC + β Ablation (Apple Full-Body) — RTX 5090

Combine a **BaseController** (heuristic + EMA from NB04) with a learned
**residual policy** via SAC. Train **5 β variants** ∈ {0.1, 0.25, 0.5, 0.75, 1.0}
and select the best. RTX 5090 optimized with 2M steps per variant.

$$a_{final} = \text{clip}(a_{base} + \beta \cdot a_{residual},\ \text{low},\ \text{high})$$

| β | Meaning |
|---|---------|
| 0.10 | Very conservative — almost entirely heuristic |
| 0.25 | Conservative — mostly follows heuristic |
| 0.50 | Balanced — moderate deviation allowed |
| 0.75 | Moderate — SAC has more influence |
| 1.00 | Aggressive — can fully override heuristic |


## Objectives

1. Re-define heuristic policy and BaseController (from NB04 pattern).
2. Build `ResidualActionWrapper`.
3. Train Residual SAC for each of 5 β values (same budget as NB05/NB06 per run).
4. Quick eval (20 episodes) for each β.
5. Ablation table: select best β by mean reward.
6. Save all models + checkpoints.


## Resources

| Resource | Requirement | Notes |
|----------|-------------|-------|
| GPU | **RTX 5090** (32 GB VRAM) | 5 training runs |
| RAM | 40 GB | 10M buffer per run |
| Runtime | ~10-20 hours | 5 × 2M steps |


## Imports & Setup


In [ ]:
import sys, os, json, random, time, copy
from pathlib import Path

import numpy as np
import torch
import gymnasium as gym
import pandas as pd
import matplotlib.pyplot as plt

from stable_baselines3 import SAC
from stable_baselines3.common.callbacks import BaseCallback, CheckpointCallback

import mani_skill.envs
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

PROJECT_ROOT = Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
from src.envs import UnitreeG1PlaceAppleInBowlFullBodyEnv

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts" / "NB07"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

IS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if IS_GPU else "cpu"

# Load SAC config from NB06 for consistency
sac_cfg_path = PROJECT_ROOT / "artifacts" / "NB06" / "nb06_config.json"
assert sac_cfg_path.exists(), "Run NB06 first!"
with open(sac_cfg_path) as f:
    sac_cfg = json.load(f)
print(f"Loaded SAC config from NB06 (total_env_steps={sac_cfg['total_env_steps']:,})")


## Configuration


In [ ]:
CFG = {
    "seed":            42,
    "env_id":          "UnitreeG1PlaceAppleInBowlFullBody-v1",
    "control_mode":    "pd_joint_delta_pos",
    "obs_mode":        "state",
    # ── Residual-specific (5 β variants for RTX 5090) ──
    "beta_values":     [0.1, 0.25, 0.5, 0.75, 1.0],
    "smooth_alpha":    0.3,
    # ── Same training budget per run as NB05/NB06 ──
    "total_env_steps": sac_cfg["total_env_steps"],
    # ── SAC hyperparams (inherited from NB06, RTX 5090) ──
    "learning_rate":   sac_cfg["learning_rate"],
    "lr_end":          sac_cfg.get("lr_end", 1e-5),
    "buffer_size":     sac_cfg["buffer_size"],
    "batch_size":      sac_cfg["batch_size"],
    "tau":             sac_cfg["tau"],
    "gamma":           sac_cfg["gamma"],
    "ent_coef":        sac_cfg["ent_coef"],
    "net_arch":        sac_cfg["net_arch"],
    "activation_fn":   sac_cfg.get("activation_fn", "ReLU"),
    "learning_starts": sac_cfg["learning_starts"],
    # ── Checkpointing ──
    "checkpoint_freq": 200_000,
    # ── Eval ──
    "eval_episodes":   20,
}

SEED = CFG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

with open(ARTIFACTS_DIR / "nb07_config.json", "w") as f:
    json.dump(CFG, f, indent=2)

print("Residual SAC Config (RTX 5090):")
print(f"  Beta values: {CFG['beta_values']} (5 variants)")
print(f"  Budget per run: {CFG['total_env_steps']:,} steps")
print(f"  net_arch: {CFG['net_arch']}, activation: {CFG['activation_fn']}")


## Step 0 — Linear LR Schedule


In [ ]:
def linear_schedule(initial_lr: float, final_lr: float = 1e-5):
    def schedule(progress_remaining: float) -> float:
        return final_lr + (initial_lr - final_lr) * progress_remaining
    return schedule

print(f"LR: {CFG['learning_rate']} → {CFG['lr_end']} (linear decay)")


## Step 1 — Heuristic Policy & BaseController

Re-define the heuristic and BaseController from NB04 pattern.


In [ ]:
def heuristic_policy(obs, env):
    """Proportional control: arm joints get small reaching signals."""
    action = np.zeros(env.action_space.shape[0], dtype=np.float32)
    arm_start, arm_end = 13, 23  # approximate arm joint range
    action[arm_start:arm_end] = np.random.uniform(-0.1, 0.1, arm_end - arm_start) * 0.5
    return action


class BaseController:
    """Heuristic + EMA smoothing for residual learning."""

    def __init__(self, env, alpha=0.3):
        self.env = env
        self.alpha = alpha
        self._prev_action = None

    def get_action(self, obs):
        raw = heuristic_policy(obs, self.env)
        if self._prev_action is None:
            self._prev_action = raw.copy()
        smoothed = self.alpha * raw + (1 - self.alpha) * self._prev_action
        self._prev_action = smoothed.copy()
        return smoothed

    def reset(self):
        self._prev_action = None

print("✅ Heuristic + BaseController defined")


## Step 2 — ResidualActionWrapper

Combines base controller action with learned residual.


In [ ]:
class ResidualActionWrapper(gym.ActionWrapper):
    """a_final = clip(a_base + beta * a_residual, low, high)

    SAC outputs the residual; this wrapper adds the base action.
    """

    def __init__(self, env, base_controller, beta=0.5):
        super().__init__(env)
        self.base_controller = base_controller
        self.beta = beta
        self._current_obs = None

    def reset(self, **kwargs):
        self.base_controller.reset()
        result = self.env.reset(**kwargs)
        if isinstance(result, tuple):
            self._current_obs = result[0]
        else:
            self._current_obs = result
        return result

    def step(self, action):
        result = self.env.step(action)
        if isinstance(result, tuple) and len(result) >= 1:
            self._current_obs = result[0]
        return result

    def action(self, residual_action):
        """Transform SAC output (residual) into final action."""
        base_action = self.base_controller.get_action(self._current_obs)
        final = base_action + self.beta * residual_action
        return np.clip(final, self.action_space.low, self.action_space.high)

print("✅ ResidualActionWrapper defined")


## Step 3 — Train Residual SAC for Each β (5 variants × 2M steps)


In [ ]:
ablation_results = []

for beta in CFG["beta_values"]:
    print(f"\n{'='*60}")
    print(f"  Training Residual SAC — β={beta}")
    print(f"  Budget: {CFG['total_env_steps']:,} steps, net={CFG['net_arch']}")
    print(f"{'='*60}")

    # Create fresh env + wrapper
    raw_env = gym.make(
        CFG["env_id"], num_envs=1, obs_mode=CFG["obs_mode"],
        control_mode=CFG["control_mode"], render_mode="rgb_array",
    )
    raw_env = CPUGymWrapper(raw_env)
    base_ctrl = BaseController(raw_env, alpha=CFG["smooth_alpha"])
    wrapped_env = ResidualActionWrapper(raw_env, base_ctrl, beta=beta)

    # LR schedule
    lr_sched = linear_schedule(CFG["learning_rate"], CFG["lr_end"])

    # Train SAC (residual)
    sac_model = SAC(
        "MlpPolicy", wrapped_env,
        learning_rate=lr_sched,
        buffer_size=CFG["buffer_size"],
        batch_size=CFG["batch_size"],
        tau=CFG["tau"],
        gamma=CFG["gamma"],
        ent_coef=CFG["ent_coef"],
        learning_starts=CFG["learning_starts"],
        policy_kwargs={
            "net_arch": CFG["net_arch"],
            "activation_fn": torch.nn.ReLU,
        },
        verbose=0,
        seed=SEED,
        device=DEVICE,
    )

    start_time = time.time()
    sac_model.learn(total_timesteps=CFG["total_env_steps"], progress_bar=True)
    train_time = time.time() - start_time

    # Save model
    model_name = f"residual_apple_beta{beta}"
    sac_model.save(str(ARTIFACTS_DIR / model_name))

    # Quick eval
    eval_rewards, eval_successes = [], []
    for ep in range(CFG["eval_episodes"]):
        obs, info = wrapped_env.reset(seed=ep * 100)
        ep_reward = 0.0
        for step in range(1000):
            action, _ = sac_model.predict(obs, deterministic=True)
            obs, reward, terminated, truncated, info = wrapped_env.step(action)
            ep_reward += float(reward)
            if terminated or truncated:
                break
        eval_rewards.append(ep_reward)
        eval_successes.append(bool(info.get("success", False)))

    ablation_results.append({
        "beta":          beta,
        "mean_reward":   float(np.mean(eval_rewards)),
        "std_reward":    float(np.std(eval_rewards)),
        "success_rate":  float(np.mean(eval_successes)),
        "training_time": train_time,
    })

    wrapped_env.close()
    print(f"  β={beta}: mean_reward={np.mean(eval_rewards):.4f}, "
          f"success={np.mean(eval_successes):.2%}, time={train_time:.0f}s")


## Step 4 — Ablation Table


In [ ]:
ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_csv(ARTIFACTS_DIR / "ablation_table.csv", index=False)

print("\nAblation Table (5 β variants):")
print(ablation_df.to_string(index=False))

# Select best beta
best_idx = ablation_df["mean_reward"].idxmax()
best_beta = float(ablation_df.loc[best_idx, "beta"])
best_info = {
    "best_beta":        best_beta,
    "best_mean_reward": float(ablation_df.loc[best_idx, "mean_reward"]),
    "best_model":       f"residual_apple_beta{best_beta}.zip",
}
with open(ARTIFACTS_DIR / "best_beta.json", "w") as f:
    json.dump(best_info, f, indent=2)

print(f"\n🏆 Best β = {best_beta} (mean_reward={best_info['best_mean_reward']:.4f})")


## Step 5 — Ablation Plot


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

betas = ablation_df["beta"].tolist()
rewards = ablation_df["mean_reward"].tolist()
stds = ablation_df["std_reward"].tolist()
successes = ablation_df["success_rate"].tolist()

x_labels = [f"β={b}" for b in betas]
colors = ["#E0E0E0", "#FFB74D", "#64B5F6", "#81C784", "#CE93D8"]

axes[0].bar(x_labels, rewards, yerr=stds, color=colors, edgecolor="black", capsize=5)
axes[0].set_title("Mean Reward by β (20 eval episodes)")
axes[0].set_ylabel("Mean Reward")
axes[0].grid(axis="y", alpha=0.3)

axes[1].bar(x_labels, successes, color=colors, edgecolor="black")
axes[1].set_title("Success Rate by β")
axes[1].set_ylabel("Success Rate")
axes[1].grid(axis="y", alpha=0.3)

fig.suptitle("NB07 — Residual SAC β Ablation (5 variants × 2M steps, RTX 5090)",
             fontweight="bold")
fig.tight_layout()
fig.savefig(ARTIFACTS_DIR / "ablation_plot.png", dpi=150)
plt.show()


## Cleanup


In [ ]:
print("✅ NB07 Residual SAC Training Complete")


## Artifacts

| File | Description |
|------|-------------|
| `residual_apple_beta0.1.zip` | Model for β=0.1 |
| `residual_apple_beta0.25.zip` | Model for β=0.25 |
| `residual_apple_beta0.5.zip` | Model for β=0.5 |
| `residual_apple_beta0.75.zip` | Model for β=0.75 |
| `residual_apple_beta1.0.zip` | Model for β=1.0 |
| `ablation_table.csv` | Performance per beta |
| `ablation_plot.png` | Comparison charts |
| `best_beta.json` | Best β selection + model path |

## RTX 5090 Optimization Notes

- **5 β variants** (was 3) — finer granularity for ablation study
- **2M steps per variant** — leverages RTX 5090 throughput
- **[512, 512] ReLU** — same architecture as NB05/NB06
- **10M replay buffer** — 40 GB RAM supports large buffers
- **Linear LR decay** (3e-4 → 1e-5) for each variant
- SAC hyperparams loaded from NB06 config to ensure fairness
- Total GPU time ≈ 5 × single training run
